# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates loading, exploring, and processing the [FAIRˆ²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) colorectal cancer dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset entities are referenced by their `@id` fields for reproducibility and clarity.

### Dataset Source
The dataset schema is defined in [Croissant JSON-LD](https://mlcommons.org/croissant/) format and accessible via:
  
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Access the dataset metadata object
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")
print(f"Published: {metadata.datePublished} | Version: {metadata.version}")

## 2. Data Overview
Review the available record sets, fields, and their `@id` fields.

In [ ]:
# List available record sets and their @ids
print("Available record sets (by '@id'):")
for rs in metadata.recordSets:
    print(f"- {rs['@id']}: {rs.get('name', '')}")

# For this dataset (as typical), there's likely just one main record set. Let's just extract and list fields.

# For demonstration, get all fields and their IDs from each record set
for rs in metadata.recordSets:
    print(f"\nRecordSet '@id': {rs['@id']}")
    print("Fields (by '@id'):")
    for field in rs.get('fields', []):
        field_id = field['@id']
        print(f"  - {field_id}: {field.get('name', '')} [{field.get('dataType', '')}]")
    print("Columns (by '@id') if present:")
    for file in rs.get('fileObjects', []):
        for col in file.get('columns', []):
            col_id = col['@id']
            print(f"  - {col_id}: {col.get('name', '')}")

## 3. Data Extraction
Load tabular data from each record set into a pandas DataFrame. Use the record set and field `@id`s discovered above.

In [ ]:
# Prepare to extract all dataset record sets into dataframes
record_set_ids = [rs['@id'] for rs in metadata.recordSets]

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} rows from record set {record_set_id}.")

# Print the available fields (columns) in the first/main record set
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"\nFields for {main_record_set_id}:")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
We will:
- Filter records based on a numeric field (e.g. age),
- Normalize the numeric field,
- Group data by a key attribute, using columns and fields referenced by their `@id`.

You'll need to confirm available numeric and grouping fields from the previous overview.

In [ ]:
# Identify numeric and groupable fields by @id (replace these with actual @ids if needed)
df = dataframes[main_record_set_id]
print("Column @ids:", df.columns.tolist())

# Let's guess possible numeric fields (e.g. 'age', 'interval', 'diagnosis_interval', etc.). Adjust as necessary.
numeric_field_id = None
possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or df[col].dtype.kind in 'if']
if possible_numeric_fields:
    numeric_field_id = possible_numeric_fields[0]
    print(f"Using numeric field: {numeric_field_id}")

# Similarly, pick a categorical field to group by (e.g. sex, anatomical_site, msi_status, etc.)
group_field_id = None
for col in df.columns:
    if any(name in col.lower() for name in ["sex", "site", "status", "location", "type"]):
        group_field_id = col
        break
if group_field_id:
    print(f"Grouping by field: {group_field_id}")

# Filter records (arbitrary threshold, adjust for the actual field)
if numeric_field_id is not None:
    threshold = 10
    mask = df[numeric_field_id] > threshold if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else [False]*len(df)
    filtered_df = df[mask]
    print(f"Filtered {len(filtered_df)} records where {numeric_field_id} > {threshold}.")

    # Normalize the numeric field
    filtered_df = filtered_df.copy()
    filtered_df[f"{numeric_field_id}_normalized"] = (
        (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) /
        filtered_df[numeric_field_id].std()
    )
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by the selected group_field_id
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = (
            filtered_df
            .groupby(group_field_id)
            [numeric_field_id]
            .agg(['mean', 'count', 'min', 'max'])
            .sort_values('mean', ascending=False)
        )
        print(f"\nAggregated '{numeric_field_id}' by '{group_field_id}':")
        print(grouped_df)

## 5. Visualization
Visualize data distributions or relationships between fields using matplotlib or seaborn.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if numeric_field_id is not None and len(filtered_df) > 0:
    plt.figure(figsize=(6,4))
    sns.histplot(filtered_df[numeric_field_id], bins=10, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()

# Boxplot by category/group (if available)
if group_field_id and numeric_field_id and group_field_id in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} grouped by {group_field_id}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook guided you through exploring, extracting, and visualizing the FAIRˆ² colorectal cancer dataset using `mlcroissant`. We:
- Programmatically listed record sets, fields, and columns by their `@id`s.
- Loaded data as DataFrames for further analysis.
- Selected numeric and grouping fields for exploratory data analysis.
- Visualized distributions and group differences.

For advanced analyses, refer to the full [Croissant specification](https://mlcommons.org/croissant/) and explore additional fields and data types using their `@id`s as demonstrated above.